In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
import json

In [2]:
df = pd.read_csv("../data/processed/after_encoding/df_encoded.csv", low_memory = False)
df.shape[1]

384

In [4]:
for col in df.columns:
    print(col)

still_operating
success_label
sentiment_score
review_sentiment_avg
price
rating_avg
review_count
downloads
upvotes
active_users
team_size
num_founders
content_rating
market_value_estimate
total_funding
funding_rounds
num_investors
revenue_monthly
synthetic_data
age_days
age_months
age_years
is_young
is_established
is_old
is_dead
survival_months
team_size_filled
burn_rate_monthly_est
remaining_investment
estimated_runway_months
funding_per_employee
funding_per_month_alive
funding_per_round
is_bootstrapped
is_well_funded
investor_density
category_success_rate
category_avg_price
category_median_price
category_price_std
category_avg_rating
category_median_reviews
category_product_count
category_avg_funding
category_avg_revenue
price_vs_category
rating_vs_category
price_deviation
price_tier_numeric
is_free
is_premium
market_saturation
is_crowded_market
competition_level
competitor_dominance
reviews_per_month
downloads_per_month
upvotes_per_month
engagement_score
rating_weighted
rating_confi

In [5]:
import numpy as np
import pandas as pd

# 1. SURVIVAL MONTHS (Target already exists, just clean it)
# Ensure no negative values and fill NaNs with age for operating companies
df['target_survival_months'] = df['survival_months'].fillna(df['age_months'])
df['target_survival_months'] = df['target_survival_months'].clip(lower=0)

# 2. GROWTH RATE (Velocity)
# Formula: Average monthly downloads, log-transformed to handle outliers
# We add +1 to avoid division by zero
df['target_growth_rate'] = df['downloads_per_month'] / (df['age_months'] + 1)
# Optional: Log transform for better regression performance
df['target_growth_rate_log'] = np.log1p(df['target_growth_rate'])

# 3. SHOULD RAISE FUNDING (Binary Classification)
# Logic: Raise if (Runway < 12 months AND NOT Bootstrapped) OR (High Growth AND High Burn)
# High Growth = Top 25% of growth rate
growth_threshold = df['target_growth_rate'].quantile(0.75)

df['target_raise_funding'] = (
    (
        (df['estimated_runway_months'] < 12) & 
        (df['is_bootstrapped'] == False) & 
        (df['still_operating'] == True) # Only alive companies need money
    ) | 
    (
        (df['target_growth_rate'] > growth_threshold) & 
        (df['burn_rate_monthly_est'] > df['revenue_monthly_filled'])
    )
).astype(int)

print("✅ Targets Created:")
print(f"- Survival Target Mean: {df['target_survival_months'].mean():.1f} months")
print(f"- Growth Target Mean: {df['target_growth_rate'].mean():.4f}")
print(f"- Funding Needed Count: {df['target_raise_funding'].sum()} startups")


✅ Targets Created:
- Survival Target Mean: 59.4 months
- Growth Target Mean: 5.7014
- Funding Needed Count: 3390 startups


In [11]:
print(df['target_raise_funding'].value_counts())

target_raise_funding
0    41203
1     3390
Name: count, dtype: int64


In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# ==========================================
# DEFINE ULTRA-CLEAN LEAK LISTS
# ==========================================

COMMON_LEAKS = [
    'success_label', 'still_operating', 'is_dead',
    'exit_status_Active', 'exit_status_Bankruptcy', 'exit_status_Failed',
    'exit_status_Operating', 'exit_status_Shut Down', 'exit_status_Shutdown',
    'exit_status_Still Active', 'exit_status_Unknown', 'exit_status_unknown',
    'failure_reason_cat_business_model', 'failure_reason_cat_competition',
    'failure_reason_cat_funding', 'failure_reason_cat_legal',
    'failure_reason_cat_market_fit', 'failure_reason_cat_multiple',
    'failure_reason_cat_no_market_fit', 'failure_reason_cat_other',
    'failure_reason_cat_ran_out_of_cash', 'failure_reason_cat_team',
    'failure_reason_cat_team_issues', 'failure_reason_cat_unknown',
    'failure_reason_cat_encoded',
    'success_reason_cat_revenue', 'success_reason_cat_unknown',
    'synthetic_data', 'data_source_synthetic', 'data_source_synthetic_augmentation'
]

TEXT_LEAKS = [c for c in df.columns if c.startswith('sf_') or c.startswith('fr_') or c.startswith('sr_')]

# All traction/outcome metrics (remove from ALL 3 models)
TRACTION_LEAKS = [
    'downloads', 'downloads_per_month',
    'active_users',
    'upvotes', 'upvotes_per_month',
    'reviews_per_month', 'review_count',
    'engagement_score',
    'rating_avg', 'rating_weighted', 'rating_confidence',
    'has_strong_reviews', 'has_strong_rating',
    'is_highly_rated', 'is_top_rated', 'rating_tier',
    'has_viral_reach', 'is_viral_reach'
]

# Derived scores (remove from ALL 3)
DERIVED_SCORES = [
    'risk_score', 'growth_score', 'market_fit_score', 'survival_score',
    'rating_adjustment', 'team_adjustment', 'funding_adjustment',
    'maturity_quality_interaction', 'quality_traction_interaction',
    'capital_efficiency', 'revenue_to_burn_ratio'
]

# Revenue/money (remove from survival/growth, keep SOME for funding)
REVENUE_LEAKS = [
    'revenue_estimated',
    'revenue_monthly_filled',
    'revenue_monthly',
    'monthly_profit_est',
    'is_profitable'
]

# Data source flags (correlated with survival)
DATA_SOURCE_LEAKS = [
    'source_encoded',
    'data_source_playstore',
    'data_source_producthunt', 
    'data_source_indiehackers',
    'data_source_failory',
    'data_source_india_unicorns'
]

# Metadata flags (data quality, not predictive)
METADATA_LEAKS = [
    'has_business_model',
    'has_founders',
    'has_brand',
    'has_website_url',
    'has_country',
    'has_city',
    'has_marketing_channels',
    'has_case_study_url',
    'info_completeness',
    'has_long_description',
    'has_detailed_name'
]

# ==========================================
# MODEL 3: SURVIVAL (ULTRA CLEAN)
# ==========================================
SURVIVAL_LEAKS = (COMMON_LEAKS + TEXT_LEAKS + TRACTION_LEAKS + 
                  DERIVED_SCORES + REVENUE_LEAKS + DATA_SOURCE_LEAKS + [
    'survival_months', 'target_survival_months',
    'age_days', 'age_months', 'age_years',
    'is_young', 'is_established', 'is_old',
    'estimated_runway_months',
    'funding_per_month_alive'  # Needs survival time
])

X_surv = df.drop(columns=SURVIVAL_LEAKS, errors='ignore')
y_surv = df['target_survival_months']

print(f"📊 SURVIVAL MODEL:")
print(f"   Features: {df.shape[1]} → {X_surv.shape[1]} (dropped {df.shape[1] - X_surv.shape[1]})")

X_train_sm, X_test_sm, y_train_sm, y_test_sm = train_test_split(
    X_surv, y_surv, test_size=0.2, random_state=42
)

X_train_sm.to_csv('../data/processed/train_test_splits/X_train_sm.csv', index=False)
X_test_sm.to_csv('../data/processed/train_test_splits/X_test_sm.csv', index=False)
y_train_sm.to_csv('../data/processed/train_test_splits/y_train_sm.csv', index=False)
y_test_sm.to_csv('../data/processed/train_test_splits/y_test_sm.csv', index=False)
print("✅ Saved CLEAN survival splits!")

# ==========================================
# MODEL 4: GROWTH (ULTRA CLEAN)
# ==========================================
GROWTH_LEAKS = (COMMON_LEAKS + TEXT_LEAKS + TRACTION_LEAKS + 
                DERIVED_SCORES + REVENUE_LEAKS + DATA_SOURCE_LEAKS + 
                METADATA_LEAKS + [
    'target_growth_rate', 'target_growth_rate_log',
    'age_days', 'age_months', 'age_years',
    'is_young', 'is_established', 'is_old',
    'target_survival_months',
    'survival_months',
    'estimated_runway_months',
    'funding_per_month_alive',
    'funding_per_round',
    'remaining_investment',
    'market_value_estimate',
    'is_well_funded'
])

X_grow = df.drop(columns=GROWTH_LEAKS, errors='ignore')
y_grow = df['target_growth_rate_log']

print(f"\n📊 GROWTH MODEL:")
print(f"   Features: {df.shape[1]} → {X_grow.shape[1]} (dropped {df.shape[1] - X_grow.shape[1]})")

X_train_gr, X_test_gr, y_train_gr, y_test_gr = train_test_split(
    X_grow, y_grow, test_size=0.2, random_state=42
)

X_train_gr.to_csv('../data/processed/train_test_splits/X_train_gr.csv', index=False)
X_test_gr.to_csv('../data/processed/train_test_splits/X_test_gr.csv', index=False)
y_train_gr.to_csv('../data/processed/train_test_splits/y_train_gr.csv', index=False)
y_test_gr.to_csv('../data/processed/train_test_splits/y_test_gr.csv', index=False)
print("✅ Saved CLEAN growth splits!")

# ==========================================
# MODEL 5: FUNDING (ULTRA CLEAN)
# ==========================================
FUNDING_LEAKS = (COMMON_LEAKS + TEXT_LEAKS + TRACTION_LEAKS + 
                 DERIVED_SCORES + DATA_SOURCE_LEAKS + [
    'target_raise_funding',
    'estimated_runway_months',  # Used in formula!
    'is_well_funded',
    'funding_adjustment',
    'funding_team_interaction',
    'funding_per_month_alive',
    'funding_per_round',
    'market_value_estimate'
])
# Note: We KEEP revenue/burn for funding (needed to assess need)

X_fund = df.drop(columns=FUNDING_LEAKS, errors='ignore')
y_fund = df['target_raise_funding']

print(f"\n📊 FUNDING MODEL:")
print(f"   Features: {df.shape[1]} → {X_fund.shape[1]} (dropped {df.shape[1] - X_fund.shape[1]})")

X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_fund, y_fund, test_size=0.2, random_state=42, stratify=y_fund
)

X_train_rf.to_csv('../data/processed/train_test_splits/X_train_rf.csv', index=False)
X_test_rf.to_csv('../data/processed/train_test_splits/X_test_rf.csv', index=False)
y_train_rf.to_csv('../data/processed/train_test_splits/y_train_rf.csv', index=False)
y_test_rf.to_csv('../data/processed/train_test_splits/y_test_rf.csv', index=False)
print("✅ Saved CLEAN funding splits!")

print("\n🔥 ALL 3 MODELS CLEANED AND SAVED!")


📊 SURVIVAL MODEL:
   Features: 388 → 209 (dropped 179)
✅ Saved CLEAN survival splits!

📊 GROWTH MODEL:
   Features: 388 → 192 (dropped 196)
✅ Saved CLEAN growth splits!

📊 FUNDING MODEL:
   Features: 388 → 217 (dropped 171)
✅ Saved CLEAN funding splits!

🔥 ALL 3 MODELS CLEANED AND SAVED!


In [3]:
with open("leakage_rules.json", "r") as f:
    leakage_report = json.load(f)

for target, leaks in leakage_report.items():
    print(f"{target}: {len(leaks)} to remove")

success_label: 21 to remove
still_operating: 6 to remove
estimated_break_even_months: 7 to remove
revenue_estimated: 7 to remove
risk_score: 4 to remove
market_fit_score: 5 to remove
failure_reason_cat: 5 to remove


In [4]:
old_key = "failure_reason_cat"
new_key = "failure_reason_cat_encoded"

if old_key in leakage_report:
    leakage_report[new_key] = leakage_report.pop(old_key)
    
print("Changed!")

Changed!


In [5]:
model_datasets = {}

TARGETS = [
    'success_label',
    'still_operating', 
    'estimated_break_even_months',
    'revenue_estimated',
    'risk_score',
    'market_fit_score',
    'failure_reason_cat_encoded'
]

for target in TARGETS:
    print(f"Target: {target} processing...")

    if target not in df.columns:
        print(f"Target: {target} not found!")
        continue

    leakage_cols = leakage_report.get(target, [])

    feature_cols = [col for col in df.columns if col not in leakage_cols and col!=target]

    X = df[feature_cols].copy()
    y = df[target].copy()

    print(f"   Original samples: {len(X)}")
    print(f"   Original features: {len(feature_cols)}")
    print(f"   Removed {len(leakage_cols)} leakage features")

    if target == 'success_label':
        valid_mask = y.isin([0, 1])
        X = X[valid_mask]
        y = y[valid_mask]
        print(f"  Filtered to binary (0/1): {len(X)} samples")
        print(f"      Class 0 (failure): {(y == 0).sum()}")
        print(f"      Class 1 (success): {(y == 1).sum()}")
    
    elif target == 'failure_reason_cat_encoded':
        if y.dtype == 'object':
            valid_mask = y != 'unknown'
            X = X[valid_mask]
            y = y[valid_mask]
            print(f"   Removed 'unknown': {len(X)} samples")

    model_datasets[target] = {
        'X': X,
        'y': y,
        'features': feature_cols,
        'n_samples': len(X),
        'n_features': len(feature_cols),
        'leakage_removed': len(leakage_cols)
        }
    
    print(f"Final: {X.shape}")

Target: success_label processing...
   Original samples: 44593
   Original features: 369
   Removed 21 leakage features
  Filtered to binary (0/1): 4196 samples
      Class 0 (failure): 1878
      Class 1 (success): 2318
Final: (4196, 369)
Target: still_operating processing...
   Original samples: 44593
   Original features: 380
   Removed 6 leakage features
Final: (44593, 380)
Target: estimated_break_even_months processing...
   Original samples: 44593
   Original features: 377
   Removed 7 leakage features
Final: (44593, 377)
Target: revenue_estimated processing...
   Original samples: 44593
   Original features: 377
   Removed 7 leakage features
Final: (44593, 377)
Target: risk_score processing...
   Original samples: 44593
   Original features: 380
   Removed 4 leakage features
Final: (44593, 380)
Target: market_fit_score processing...
   Original samples: 44593
   Original features: 379
   Removed 5 leakage features
Final: (44593, 379)
Target: failure_reason_cat_encoded processing

In [6]:
for target, data in model_datasets.items():
    X = data['X']
    y = data['y']
    print(f"{target}: {len(X)}")

success_label: 4196
still_operating: 44593
estimated_break_even_months: 44593
revenue_estimated: 44593
risk_score: 44593
market_fit_score: 44593
failure_reason_cat_encoded: 44593


In [16]:
imb_models = ["success_label", "still_operating", "failure_reason_cat_encoded"]

X_train_dict = {}
X_test_dict = {}
y_train_dict = {}
y_test_dict = {}

for target, data in model_datasets.items():
    X = data['X']
    y = data['y']

    if target == 'failure_reason_cat_encoded':
            print(f"   Original samples: {len(X)}")
            valid_mask = y != 0  
            X = X[valid_mask]
            y = y[valid_mask]
            print(f"   Removed 'unknown' (class 0): {len(X)} samples remaining")
            print(f"   Classes: {y.nunique()} (0-12 → 1-12)")
        
    if target in imb_models:
        print(f"Classification model: {target}")
        print("<Train, test split with stratified y>")
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True, stratify=y)

        if target == 'still_operating':
            print(f"   Applying SMOTETomek (conservative)...")
            smt = SMOTETomek(random_state=42)
            X_train_res, y_train_res = smt.fit_resample(X_train, y_train)

        if target == "failure_reason_cat_encoded":
            print(f" New train size: X={len(X_train)}, y={len(y_train)}")
            print(f" Test size unchanged: X={len(X_test)}, y={len(y_test)}")
            X_train.to_csv(f'../data/processed/train_test_splits/X_train_{target}.csv', index=False)
            X_test.to_csv(f'../data/processed/train_test_splits/X_test_{target}.csv', index=False)
            y_train.to_csv(f'../data/processed/train_test_splits/y_train_{target}.csv', index=False)
            y_test.to_csv(f'../data/processed/train_test_splits/y_test_{target}.csv', index=False)
            break
       
        else:
            print(f"   Applying SMOTE...")
            sm = SMOTE(random_state=42, sampling_strategy='auto')
            X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
        
        print(f"   After SMOTE: {len(X_train_res)} samples")
        print(f"   Class distribution:")
        unique, counts = np.unique(y_train_res, return_counts=True)
        for cls, cnt in zip(unique, counts):
            print(f"      Class {cls}: {cnt}")

        X_train_dict[target] = X_train_res
        X_test_dict[target] = X_test
        y_train_dict[target] = y_train_res
        y_test_dict[target] = y_test
        
        print(f"  SMOTE applied! New train size: X={len(X_train_res)}, y={len(y_train_res)}")
        print(f"  Test size unchanged: X={len(X_test)}, y={len(y_test)}")
        X_train_res.to_csv(f'../data/processed/train_test_splits/X_train_res_{target}.csv', index=False)
        X_test.to_csv(f'../data/processed/train_test_splits/X_test_{target}.csv', index=False)
        y_train_res.to_csv(f'../data/processed/train_test_splits/y_train_res_{target}.csv', index=False)
        y_test.to_csv(f'../data/processed/train_test_splits/y_test_{target}.csv', index=False)
        print(" Classification Saved!")
        
    else:
        print(f"Regression model: {target}")
        print("<Train, test split without stratified y>")
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)
        
        X_train_dict[target] = X_train
        X_test_dict[target] = X_test
        y_train_dict[target] = y_train
        y_test_dict[target] = y_test
        
        print(f"  Completed {target}: X train - {len(X_train)}, y train - {len(y_train)}, X test - {len(X_test)}, y test - {len(y_test)}")
        X_train.to_csv(f'../data/processed/train_test_splits/X_train_{target}.csv', index=False)
        X_test.to_csv(f'../data/processed/train_test_splits/X_test_{target}.csv', index=False)
        y_train.to_csv(f'../data/processed/train_test_splits/y_train_{target}.csv', index=False)
        y_test.to_csv(f'../data/processed/train_test_splits/y_test_{target}.csv', index=False)
        print(" Regression model Saved!")

print("Saved All Splits!")

Classification model: success_label
<Train, test split with stratified y>
   Applying SMOTE...
   After SMOTE: 3708 samples
   Class distribution:
      Class 0: 1854
      Class 1: 1854
  SMOTE applied! New train size: X=3708, y=3708
  Test size unchanged: X=840, y=840
 Classification Saved!
Classification model: still_operating
<Train, test split with stratified y>
   Applying SMOTETomek (conservative)...
   Applying SMOTE...
   After SMOTE: 63968 samples
   Class distribution:
      Class False: 31984
      Class True: 31984
  SMOTE applied! New train size: X=63968, y=63968
  Test size unchanged: X=8919, y=8919
 Classification Saved!
Regression model: estimated_break_even_months
<Train, test split without stratified y>
  Completed estimated_break_even_months: X train - 35674, y train - 35674, X test - 8919, y test - 8919
 Regression model Saved!
Regression model: revenue_estimated
<Train, test split without stratified y>
  Completed revenue_estimated: X train - 35674, y train - 3567

In [8]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import os

save_dir = '../data/processed/train_test_splits/'
os.makedirs(save_dir, exist_ok=True)

targets = [
    'success_label', 'still_operating', 'estimated_break_even_months',
    'revenue_estimated', 'risk_score', 'market_fit_score', 
    'failure_reason_cat_encoded'
]

for target in targets:
    print(f"\nProcessing {target}...")
    
    if target in ["success_label", "still_operating", "failure_reason_cat_encoded"]:
        train_file = f"X_train_res_{target}.csv"
    else:
        train_file = f"X_train_{target}.csv" 
        
    test_file = f"X_test_{target}.csv"
 
    if not os.path.exists(save_dir + train_file):
        print(f"⚠️ File not found: {train_file}, skipping...")
        continue

    # 2. Load Data
    X_train = pd.read_csv(save_dir + train_file)
    X_test = pd.read_csv(save_dir + test_file)
    
    # 3. Initialize Scaler
    scaler = StandardScaler()
    
    # 4. FIT on TRAIN, TRANSFORM on TEST
    # (Never fit on test, that is data leakage!)
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)
    
    # 5. Save Scaled Versions
    X_train_scaled.to_csv(save_dir + f"X_train_scaled_{target}.csv", index=False)
    X_test_scaled.to_csv(save_dir + f"X_test_scaled_{target}.csv", index=False)
    
    print(f"✅ Scaled {target}. Mean should be ~0, Std ~1.")

print("\n🚀 ALL DATA SCALED. READY FOR MODEL TRAINING.")


Processing success_label...
✅ Scaled success_label. Mean should be ~0, Std ~1.

Processing still_operating...
✅ Scaled still_operating. Mean should be ~0, Std ~1.

Processing estimated_break_even_months...
✅ Scaled estimated_break_even_months. Mean should be ~0, Std ~1.

Processing revenue_estimated...
✅ Scaled revenue_estimated. Mean should be ~0, Std ~1.

Processing risk_score...
✅ Scaled risk_score. Mean should be ~0, Std ~1.

Processing market_fit_score...
✅ Scaled market_fit_score. Mean should be ~0, Std ~1.

Processing failure_reason_cat_encoded...


MemoryError: Unable to allocate 795. MiB for an array with shape (258, 403836) and data type float64